In [1]:
# 03_representative_cases.ipynb
# Stages 2-3 on single representative Class 3 patients (one male, one female),
# using the three-layer guardrail structure from guardrail_core.py:
#   Layer 1 (safety)     : absolute floors + anthropometric coupling, code-enforced
#   Layer 2 (discretion) : delegated to the GPT-4o-mini agent (or to the rule
#                          baseline), the layer where LLM and rule may differ
#   Layer 3 (fixed)      : non-modifiable variables, held at baseline
#
# Requires an OpenAI API key for the LLM guardrail (Stage 2). Set it first:
#     import os; os.environ["OPENAI_API_KEY"] = "sk-..."
# The rule-based guardrail runs without any API access.
#
# Inputs  (../data/): df_final.pkl, agent_config.pkl, model_male.pkl, model_female.pkl
# Outputs (../results/tables/): representative_cases.json

import os
import json
import joblib
import numpy as np
import pandas as pd
import openai
import dice_ml

import guardrail_core as gc
import experiment_core as ec

DATA = os.path.join("..", "data")
TAB_DIR = os.path.join("..", "results", "tables")
os.makedirs(TAB_DIR, exist_ok=True)

_api_key = os.environ.get("OPENAI_API_KEY", "")
client = openai.OpenAI(api_key=_api_key) if _api_key else None

df_final = joblib.load(os.path.join(DATA, "df_final.pkl"))
ac = joblib.load(os.path.join(DATA, "agent_config.pkl"))
model_male = joblib.load(os.path.join(DATA, "model_male.pkl"))
model_female = joblib.load(os.path.join(DATA, "model_female.pkl"))
X = ac["X_features"]; T = ac["target_col"]; VARY = ac["vary_features"]


def llm_guardrail_raw(patient_row, gender_name):
    """GPT-4o-mini proposes a permitted_range (Layer-2 discretion)."""
    info = {k: float(patient_row[k]) for k in X}
    prompt = f"""
You are a high-precision Clinical Data Strategist for a health insurer.
Respond ONLY in valid JSON using the exact variable names given.

[Patient ({gender_name})]
{json.dumps(info, ensure_ascii=False, indent=1)}

[Variables]
{", ".join(X)}

[Principles]
- Propose a permitted_range [min, max] per variable, tailored to THIS patient.
- Reductions only for BMI/WaistCirc/Weight; keep them mutually consistent.
- Cap Carb_g and Sugar_g at current when Sodium_mg is reduced.
- Keep Energy_kcal at or below current.

[Format - valid JSON only]
{{"reasoning": "...", "guardrail_ranges": {{"Var": [min, max]}}}}
"""
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": "Respond strictly in valid JSON."},
                  {"role": "user", "content": prompt}],
        response_format={"type": "json_object"})
    return json.loads(resp.choices[0].message.content)


def representative_case(model, gender_code, gender_name, use_llm=True):
    df_stable = df_final[df_final["Sex"] == gender_code].copy().astype(float)
    pool = df_stable[(df_stable["Sex"] == gender_code) &
                     (df_stable[T] == 3.0) & (df_stable["BMI"] >= 23.0)].copy()
    preds = model.predict(pool[X])
    confirmed = pool[preds == 3]
    if len(confirmed) == 0:
        confirmed = pool
    q = confirmed.sample(1, random_state=42)[X]
    cur = {f: float(q.iloc[0][f]) for f in X}
    print(f"\n{gender_name}: BMI={cur['BMI']:.1f}, Sodium={cur['Sodium_mg']:.0f}, "
          f"Energy={cur['Energy_kcal']:.0f}")

    # Rule guardrail (aggressive delegation)
    g_rule = gc.build_rule_guardrails(cur, X, "aggressive")

    # LLM guardrail (Layer-2 discretion) + Layer-1 enforcement
    reasoning = ""
    if use_llm and os.environ.get("OPENAI_API_KEY"):
        plan = llm_guardrail_raw(q.iloc[0], gender_name)
        reasoning = plan.get("reasoning", "")
        g_llm = gc.sanitize_llm_guardrails(plan.get("guardrail_ranges", {}),
                                           cur, X, "aggressive")
    else:
        print("  (no API key: using rule guardrail as LLM stand-in)")
        g_llm = g_rule

    # Run DiCE under each guardrail with stepwise fallback
    exp = ec.make_dice(model, df_stable, X, T)
    r_llm = ec.eval_pre_injection(exp, q, df_stable, X, VARY, g_llm)
    r_rule = ec.eval_pre_injection(exp, q, df_stable, X, VARY, g_rule)
    r_pure = ec.eval_C0_pure(exp, q, df_stable, X, VARY)

    print(f"  LLM guardrail  -> feasible={r_llm['feasible']}, "
          f"class={r_llm['achieved_class']}, changed={r_llm['n_changed_vars']}")
    print(f"  Rule guardrail -> feasible={r_rule['feasible']}, "
          f"class={r_rule['achieved_class']}, changed={r_rule['n_changed_vars']}")
    print(f"  Pure DiCE      -> feasible={r_pure['feasible']}, "
          f"any_viol={r_pure['any_viol']}, changed={r_pure['n_changed_vars']}")

    return {
        "gender": gender_name,
        "patient": cur,
        "llm_reasoning": reasoning,
        "guardrail_rule": g_rule,
        "guardrail_llm": g_llm,
        "result_llm": {k: (None if (isinstance(v, float) and np.isnan(v)) else v)
                       for k, v in r_llm.items()},
        "result_rule": {k: (None if (isinstance(v, float) and np.isnan(v)) else v)
                        for k, v in r_rule.items()},
        "result_pure": {k: (None if (isinstance(v, float) and np.isnan(v)) else v)
                        for k, v in r_pure.items()},
    }


male_case = representative_case(model_male, 1.0, "Male", use_llm=True)
female_case = representative_case(model_female, 2.0, "Female", use_llm=True)

with open(os.path.join(TAB_DIR, "representative_cases.json"), "w", encoding="utf-8") as f:
    json.dump({"male": male_case, "female": female_case}, f,
              ensure_ascii=False, indent=2, default=float)
print("\nsaved -> representative_cases.json")



Male: BMI=26.2, Sodium=5170, Energy=1414
  (no API key: using rule guardrail as LLM stand-in)


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.76it/s]


  LLM guardrail  -> feasible=True, class=0, changed=16.0
  Rule guardrail -> feasible=True, class=0, changed=13.67
  Pure DiCE      -> feasible=True, any_viol=1, changed=16.0

Female: BMI=40.1, Sodium=2699, Energy=1006
  (no API key: using rule guardrail as LLM stand-in)


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.64it/s]

  LLM guardrail  -> feasible=True, class=1, changed=15.25
  Rule guardrail -> feasible=True, class=1, changed=13.0
  Pure DiCE      -> feasible=True, any_viol=1, changed=15.75

saved -> representative_cases.json
